In [1]:
from qiskit import *
import numpy as np
#[j=0]     H2molecule     Fixed-Duratuion  2026-8-1
n_site=2  
N=n_site 
j_coupling=1
a3=-0.0115
a0=0
a00=-1.04391
a2=-0.42045
J=0.42045 
a1=J
a4=0.179005

step_max=288*3
t_max=36
dt=t_max/step_max


### simulator and Shots
simulator=Aer.get_backend('qasm_simulator')
n_shots=10000    #10^7

##### define quantum circuit
qr=QuantumRegister(14,'qr')
cr=ClassicalRegister(14,'cr')
qc=QuantumCircuit(qr, cr)

### measurement
qc_meas=QuantumCircuit(qr,cr)
qc_meas.measure(qr,cr)

### LIst
time_list_qiskit=[]
mag_list_qiskit=[]
magx_list_qiskit=[]
magy_list_qiskit=[]

#####  Important : Initial state    |+\rangle|-\rangle

#qc.x(qr[13]) 
#qc.x(qr[12])     

qc.h(qr[13])
qc.h(qr[12])

#################################   Z:measurement
  
qc_total=qc.compose(qc_meas)

job=execute(qc_total, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total)

############################
mag00=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                           
        obs=obs+1
        if j>=n_site-2:                                     
            mag00=mag00+a0*obs*count/n_shots
        else:
            mag00=mag00

print('mag00=',mag00)

    
mag=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j]))  
        if j>=n_site-2:                    
            mag=mag+a1*obs*count/n_shots/(1)
        else:
            mag=mag

mag_list_qiskit.append(mag)
print('mag=',mag)
#################################
mag2=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j+1]))  
        if j>=n_site-2:                    
            mag2=mag2+a2*obs*count/n_shots/(1)
        else:
            mag2=mag2
#time_list_qiskit.append(dt*step)
#mag_list_qiskit.append(mag)
print('mag2=',mag2)

############################################  measurement:XX-energy
qc.h(qr[12])
qc.h(qr[13])
qc_total_xx=qc.compose(qc_meas)
qc.h(qr[12])
qc.h(qr[13])
    

job=execute(qc_total_xx, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_xx)
       
magx=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*count
      
    magx=a4*obs/n_shots
print('magx=', magx)

#################################
    
qc_total_zz=qc.compose(qc_meas)
  

job=execute(qc_total_zz, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_zz)
       
magzz=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*count
      
    magzz=a3*obs/n_shots
print('magzz=', magzz)

##################################################
print('EE=',mag00+magzz+mag+mag2+magx)
print('E0=', a00+magzz+mag+mag2+magx)


mag00= 0.0
mag= -0.0031954200000000044
mag2= 0.002438609999999994
magx= 0.179005
magzz= -2.07e-05
EE= 0.17822749
E0= -0.86568251


In [2]:
#[Loop]  
from functools import reduce
from operator import mul

E=0.224399      # substutute EE=-**-a00

ss=0.5*np.pi/(E)
Dt=0.01

for k in range(12):
#    if k==6:
#        E=-0.829
#ss=0.5*np.pi/(E)
    print("n=",k+1)
    qc.h(qr[11-k])

#################################################
    qc.s(qr[11-k])

    for step1A in range(100):
        qc.p(-1*a0*ss*Dt,qr[4]) 
### JZ                                                
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
        qc.cx(qr[11-k],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[11-k],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])
    
### Z1                                               
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[11-k],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[11-k],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
### ZZ

        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[11-k],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[11-k],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[13],qr[12])
    
    
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[11-k],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[11-k],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cx(qr[12],qr[13]) 

### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[11-k],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[11-k],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cx(qr[12],qr[13]) 
    

### ZZ
    
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[11-k],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[11-k],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[13],qr[12]) 
    
### Z1                                              
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[11-k],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[11-k],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        
### JZ                                               
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
        qc.cx(qr[11-k],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[11-k],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
######################################################################
        

######################################################
    
    qc.h(qr[11-k])


######################################### 
    qc_total=qc.compose(qc_meas)

    job=execute(qc_total, simulator, shots=n_shots)
    result=job.result()
    counts=result.get_counts(qc_total)

    zero=0
    for bit_string, count in counts.items():
        zero=zero+((reduce(mul,((1-int(bit_string[i])) for i in range(2,3+k)))))*count 
    print('zero=', zero)

###########################################
    mag00=0
    for bit_string, count in counts.items():
        obs=0
        for j in range(n_site-1):                           
            obs=obs+1*((reduce(mul,((1-int(bit_string[i])) for i in range(2,3+k)))))
            if j>=n_site-2:                                     
                mag00=mag00+a0*obs*count/(zero)
            else:
                mag00=mag00

    print('mag00=',mag00)
#############################################
    
    mag=0
    for bit_string, count in counts.items():
        obs=0
        for j in range(n_site-1):                               
            obs=obs+(-1)**(int(bit_string[j]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,3+k)))))
            if j>=n_site-2:                                     
                mag=mag+a1*obs*count/(zero)/(1)
            else:
                mag=mag
    mag_list_qiskit.append(mag)
    print('mag=',mag)

#############################################
    
    mag2=0
    for bit_string, count in counts.items():
        obs=0
        for j in range(n_site-1):                               
            obs=obs+(-1)**(int(bit_string[j+1]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,3+k)))))  
            if j>=n_site-2:                                   
                mag2=mag2+a2*obs*count/(zero)/(1)
            else:
                mag2=mag2
    print('mag2=',mag2)

############################################  XX-energy
    qc.h(qr[12])
    qc.h(qr[13])
    qc_total_xx=qc.compose(qc_meas)
  
    qc.h(qr[12])
    qc.h(qr[13])
    

    job=execute(qc_total_xx, simulator, shots=n_shots)
    result=job.result()
    counts=result.get_counts(qc_total_xx)
       
    magx=0
    obs=0
    for bit_string, count in counts.items():
        for j in range(1):
            obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,3+k)))))*count #*(1-int(bit_string[2]))*(1-int(bit_string[3]))*(1-int(bit_string[4]))*(1-int(bit_string[5]))*count
        magx=a4*obs/(zero)
    print('magx=', magx)

#################################
    
    qc_total_zz=qc.compose(qc_meas)
 
    job=execute(qc_total_zz, simulator, shots=n_shots)
    result=job.result()
    counts=result.get_counts(qc_total_zz)
       
    magzz=0
    obs=0
    for bit_string, count in counts.items():
        for j in range(1):
            obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,3+k)))))*count 
        magzz=a3*obs/(zero)
    print('magzz=', magzz)

########################################
    print('EE=', mag00+magzz+mag+mag2+magx)
    print('E0=', a00+magzz+mag+mag2+magx)
    print()








n= 1
zero= 7397
mag00= 0.0
mag= -0.027340333919156412
mag2= -0.04120944301743951
magx= 0.1741650648911721
magzz= -0.0034249695822630793
EE= 0.1021903183723131
E0= -0.9417196816276867

n= 2
zero= 6041
mag00= 0.0
mag= -0.047814790597583194
mag2= -0.04015885614964409
magx= 0.1682486988909121
magzz= -0.006287783479556364
EE= 0.07398726866412844
E0= -0.9699227313358713

n= 3
zero= 5159
mag00= 0.0
mag= -0.03903771079666603
mag2= -0.03968969761581699
magx= 0.16599339406861793
magzz= -0.00829453382438457
EE= 0.07897145183175035
E0= -0.9649385481682494

n= 4
zero= 4739
mag00= 0.0
mag= -0.025462998522895124
mag2= -0.038771185904199194
magx= 0.16514240557079554
magzz= -0.009053914327917282
EE= 0.09185430681578394
E0= -0.9520556931842161

n= 5
zero= 4477
mag00= 0.0
mag= -0.023947900379718553
mag2= -0.024887033727942817
magx= 0.16756979115479118
magzz= -0.009755863301317847
EE= 0.10897899374581196
E0= -0.9349310062541879

n= 6
zero= 4137
mag00= 0.0
mag= -0.012297425670775913
mag2= -0.01941163886874